# Gemma 4 E2B-IT Bio-Safety SAE Training — Colab T4

This notebook implements domain-specific fine-tuning for a TopK Sparse Autoencoder (SAE) on the Gemma 4 E2B-IT model. The goal is to enhance the SAE's representation of biosecurity concepts, specifically distinguishing between benign biological content and hazardous biological knowledge.

**Specifications:**
- **Model:** Gemma 4 E2B-IT (frozen, NF4 4-bit quantization).
- **SAE:** TopK (d_model=1536, d_sae=6144, k=32).
- **Loss:** L = L_recon + λ_sparsity * L_sparsity + λ_contrastive * L_contrastive.
- **Contrastive Objective:** Discriminates between hazard-adjacent and benign biological activations.

**Dataset options (cell [5/7]):**

| Option | Dataset | Tier coverage | Access |
|--------|---------|--------------|--------|
| `"wmdp"` (default) | `cais/wmdp-corpora` — WMDP Machine Unlearning Corpora (Li et al. 2024) | hazard + benign (~7.5K docs) | Public |
| `"gated"` | `SolshineCode/biorefusalaudit-gated` — BioRefusalAudit curated eval set | hazard + dual-use + benign (75→10K) | HF gated |
| `"local"` | Local `data/eval_set_public/eval_set_public_v1.jsonl` | all tiers (75 prompts) | Always available |

**Setup:**
1. Change runtime to **T4 GPU**.
2. Add `HF_TOKEN` (write scope) and `WANDB_API_KEY` to Colab 🔑 Secrets.
   - `HF_TOKEN` is required only for the `"gated"` dataset option and checkpoint saves to HF; not needed for the default `"wmdp"` path.
3. Toggle notebook access for both secrets.

In [ ]:
# [1/7] GPU + Environment Check
import torch
import subprocess

r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout[:1000] if r.returncode == 0 else "NO GPU found.")

assert torch.cuda.is_available(), "CUDA not available â€” set Runtime to T4 GPU"
print("\nTorch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

In [ ]:
# [2/7] Install Dependencies + Auth
!pip install -q -U 'transformers==5.5.4' accelerate bitsandbytes datasets huggingface_hub wandb sentencepiece

import os, json, time
from google.colab import userdata
from huggingface_hub import login, HfApi
import wandb

for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except:
        print(f"{key}: MISSING â€” add via ðŸ”‘ Secrets for full functionality")

if os.environ.get("HF_TOKEN"): login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
if os.environ.get("WANDB_API_KEY"): wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
# [3/7] Load Frozen Gemma 4 E2B-IT
import torch.nn as nn
from transformers import AutoTokenizer, BitsAndBytesConfig

HF_MODEL = "google/gemma-4-E2B-it"
LAYER = 17
D_MODEL = 1536

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"Loading {HF_MODEL}...")
# Gemma 4 is a multimodal model; try CausalLM first, fall back to ImageTextToText
try:
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
except Exception:
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(
        HF_MODEL, quantization_config=bnb_config,
        device_map={"":0}, low_cpu_mem_usage=True
    )
model.eval()
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)
tokenizer.pad_token = tokenizer.eos_token

class ActivationHook:
    def __init__(self): self.last = None
    def __call__(self, module, inputs, outputs):
        # Overwrite (not append) to avoid VRAM accumulation during generation
        self.last = outputs[0].detach() if isinstance(outputs, tuple) else outputs.detach()

def pick_layer(m, layer_idx):
    """Handles both causal LM (model.model.layers) and multimodal (language_model.model.layers)."""
    if hasattr(m, "language_model"):
        return m.language_model.model.layers[layer_idx]
    return m.model.layers[layer_idx]

hook = ActivationHook()
handle = pick_layer(model, LAYER).register_forward_hook(hook)
print(f"Attached hook at layer {LAYER}")


In [ ]:
# [4/7] TopKSAE Module
import torch.nn.functional as F

class TopKSAE(nn.Module):
    """TopK SAE compatible with biorefusalaudit/models/sae_adapter.py"""
    def __init__(self, d_model: int, d_sae: int, k: int):
        super().__init__()
        self.d_model, self.d_sae, self.k = d_model, d_sae, k
        
        # Weights initialization
        self.W_enc = nn.Parameter(torch.randn(d_model, d_sae) * 0.02)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_model) * 0.02)
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        
        self.normalize_decoder()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # Canonical implementation from sae_adapter.py
        pre = x @ self.W_enc + self.b_enc
        pre_relu = torch.relu(pre)
        if self.k >= self.d_sae: return pre_relu
        
        topk_vals, topk_idx = torch.topk(pre_relu, self.k, dim=-1)
        out = torch.zeros_like(pre_relu)
        out.scatter_(-1, topk_idx, topk_vals)
        return out

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return z @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        z = self.encode(x)
        x_hat = self.decode(z)
        # Return (x_hat, z, pre_activations) for loss calculation
        return x_hat, z, (x @ self.W_enc + self.b_enc)

    @torch.no_grad()
    def normalize_decoder(self):
        n = self.W_dec.norm(dim=1, keepdim=True).clamp_min(1e-8)
        self.W_dec.data.div_(n)

    @torch.no_grad()
    def project_grad(self):
        if self.W_dec.grad is None: return
        g, w = self.W_dec.grad, self.W_dec.data
        g.sub_((g * w).sum(dim=1, keepdim=True) * w)

D_SAE = 6144
K = 32
sae = TopKSAE(D_MODEL, D_SAE, K).cuda().float()

In [ ]:
# [5/7] Configurable Dataset Loading
from datasets import load_dataset, Dataset as HFDataset, concatenate_datasets

# ======================================================================
# DATASET CONFIGURATION
# ======================================================================
# Option A (DEFAULT): cais/wmdp-corpora — WMDP Machine Unlearning Corpora
#   (Li et al. 2024, arXiv:2403.03218). Publicly available on HuggingFace.
#   bio_forget_corpus → hazard_adjacent_category (N~3.9K docs)
#   bio_retain_corpus → benign_bio (N~3.7K docs)
#   Structural fit: these are the exact paired corpora designed to contrast
#   hazardous vs. safe biology knowledge — ideal for contrastive SAE training.
#
# Option B: SolshineCode/biorefusalaudit-gated — curated 75-prompt eval set
#   with native BioRefusalAudit tier labels. Requires HF approval (gated).
#   Use this once you have access to the full institutional corpus (~10K).
#
# Set DATASET_SOURCE = "wmdp" | "gated" | "local"
# ======================================================================

DATASET_SOURCE = "wmdp"   # <-- change to "gated" once biorefusalaudit-gated is accessible

HF_TEXT_COLUMN  = "prompt"
HF_LABEL_COLUMN = "tier"

dataset = None

if DATASET_SOURCE == "wmdp":
    try:
        print("Loading cais/wmdp-corpora (bio_forget_corpus + bio_retain_corpus)...")
        ds_hazard = load_dataset("cais/wmdp-corpora", "bio_forget_corpus", split="train")
        ds_benign = load_dataset("cais/wmdp-corpora", "bio_retain_corpus", split="train")

        # Synthesize tier label and normalize text column name
        ds_hazard = ds_hazard.map(lambda x: {"prompt": x["text"], "tier": "hazard_adjacent_category"},
                                  remove_columns=ds_hazard.column_names)
        ds_benign = ds_benign.map(lambda x: {"prompt": x["text"], "tier": "benign_bio"},
                                  remove_columns=ds_benign.column_names)

        # Cap at 5K per tier to balance classes; shuffle for training diversity
        MAX_PER_TIER = 5000
        ds_hazard = ds_hazard.select(range(min(MAX_PER_TIER, len(ds_hazard))))
        ds_benign = ds_benign.select(range(min(MAX_PER_TIER, len(ds_benign))))
        dataset = concatenate_datasets([ds_hazard, ds_benign]).shuffle(seed=42)

        n_h = sum(1 for r in dataset if "hazard" in r["tier"])
        n_b = sum(1 for r in dataset if "benign" in r["tier"])
        print(f"Loaded {len(dataset)} examples: {n_h} hazard-adjacent, {n_b} benign-bio")
        HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"
    except Exception as e:
        print(f"WMDP-corpora load failed ({e}); falling back to local eval set.")
        DATASET_SOURCE = "local"

elif DATASET_SOURCE == "gated":
    HF_DATASET_REPO = "SolshineCode/biorefusalaudit-gated"
    try:
        print(f"Loading {HF_DATASET_REPO}...")
        dataset = load_dataset(HF_DATASET_REPO, split="train")
        print(f"Loaded {len(dataset)} examples.")
    except Exception as e:
        print(f"Gated dataset unavailable ({e}); falling back to local eval set.")
        DATASET_SOURCE = "local"

if DATASET_SOURCE == "local" or dataset is None:
    import json as _json
    _rows = [_json.loads(l) for l in open("data/eval_set_public/eval_set_public_v1.jsonl")]
    dataset = HFDataset.from_list(_rows)
    HF_TEXT_COLUMN, HF_LABEL_COLUMN = "prompt", "tier"
    print(f"Using local public eval set: {len(dataset)} examples.")
    print("NOTE: 75-prompt fallback active — L_contrastive will plateau; use wmdp or gated option for real training.")

def sample_batch(ds, size=4):
    indices = torch.randint(0, len(ds), (size,))
    return [ds[int(i)] for i in indices]


In [ ]:
# [6/7] Training Loop Logic
def compute_contrastive_loss(z_hazard, z_benign, margin=1.0):
    """Encourage separation between hazard-adjacent and benign biological feature profiles."""
    if z_hazard.size(0) == 0 or z_benign.size(0) == 0: return torch.tensor(0.0).to(z_hazard.device)
    
    # Aggregate across sequence tokens
    m_hazard = z_hazard.mean(0)
    m_benign = z_benign.mean(0)
    
    # Minimize similarity between the two profiles
    cos_sim = F.cosine_similarity(m_hazard.unsqueeze(0), m_benign.unsqueeze(0))
    return cos_sim.mean()

def track_semantic_drift(sae, tokenizer, features_to_watch=[100, 500, 1000]):
    """Placeholder for semantic drift tracking: identifies top tokens for specific features."""
    # In a real run, this would collect activations across a calibration set.
    return {f"feat_{idx}_drift": 0.0 for idx in features_to_watch}

In [ ]:
# [7/7] Main Execution Loop
from tqdm.auto import tqdm

RUN_TAG = "gemma4-e2b-bio-sae-v1"
HF_OUTPUT_REPO = f"Solshine/{RUN_TAG}"
LR = 3e-4
LAMBDA_SPARSITY = 0.04
LAMBDA_CONTRASTIVE = 0.1
BATCH_SIZE = 4
MAX_STEPS = 2000
SAVE_EVERY = 500

optimizer = torch.optim.AdamW(sae.parameters(), lr=LR)

wandb.init(project="biorefusalaudit-sae-training", name=RUN_TAG, config={
    "model": HF_MODEL, "layer": LAYER, "d_sae": D_SAE, "k": K,
    "lr": LR, "lambda_sparsity": LAMBDA_SPARSITY, "lambda_contrastive": LAMBDA_CONTRASTIVE
})

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    batch = sample_batch(dataset, BATCH_SIZE)
    
    all_activations = []
    hazard_indices = []
    benign_indices = []
    offset = 0
    
    for item in batch:
        prompt = item[HF_TEXT_COLUMN]
        tier = item.get(HF_LABEL_COLUMN, "benign_bio")
        
        # Generate completions and capture activations for all generated tokens
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            # Capture activations during generation (simplified to re-run full sequence)
            out = model.generate(**inputs, max_new_tokens=32, do_sample=True, temperature=0.7)
            _ = model(out)
        
        acts = hook.last # (1, seq, d_model)
        seq_len = acts.size(1)
        all_activations.append(acts.squeeze(0))
        
        token_range = range(offset, offset + seq_len)
        if "hazard" in tier: hazard_indices.extend(token_range)
        elif "benign" in tier: benign_indices.extend(token_range)
        offset += seq_len
        
    # Move activations to GPU/Float32 for SAE
    x = torch.cat(all_activations, dim=0).cuda().float()
    x_hat, z, pre_relu = sae(x)
    
    # Loss 1: Reconstruction (MSE)
    l_recon = F.mse_loss(x_hat, x)
    
    # Loss 2: Sparsity (L1 on pre-activations)
    l_sparsity = pre_relu.abs().mean()
    
    # Loss 3: Contrastive (Tier separation)
    l_contrast = torch.tensor(0.0).cuda()
    if hazard_indices and benign_indices:
        l_contrast = compute_contrastive_loss(z[hazard_indices], z[benign_indices])
        
    total_loss = l_recon + LAMBDA_SPARSITY * l_sparsity + LAMBDA_CONTRASTIVE * l_contrast
    
    optimizer.zero_grad()
    total_loss.backward()
    sae.project_grad()
    optimizer.step()
    sae.normalize_decoder()
    
    # Metrics
    l0 = (z > 0).float().sum(-1).mean().item()
    drift = track_semantic_drift(sae, tokenizer)
    
    log_dict = {
        "step": step,
        "total_loss": total_loss.item(),
        "l_recon": l_recon.item(),
        "l_sparsity": l_sparsity.item(),
        "l_contrastive": l_contrast.item(),
        "l0": l0
    }
    log_dict.update(drift)
    wandb.log(log_dict)
    
    pbar.set_postfix({"loss": f"{total_loss.item():.4f}", "L0": f"{l0:.1f}"})
    
    if (step + 1) % SAVE_EVERY == 0:
        torch.save(sae.state_dict(), "sae_weights.pt")
        try:
            api = HfApi()
            api.create_repo(HF_OUTPUT_REPO, exist_ok=True)
            api.upload_file(path_or_fileobj="sae_weights.pt", 
                            path_in_repo=f"sae_weights_step_{step+1}.pt", 
                            repo_id=HF_OUTPUT_REPO)
            print(f"\nCheckpoint saved to HF: {HF_OUTPUT_REPO}")
        except Exception as e: print(f"\nHF Push failed: {e}")

handle.remove()
wandb.finish()
print("Training Complete.")